# Data quality profile

This notebook profiles the unchanged FDNY EMS source pages before cleaning. It checks missing values, duplicate incident IDs, timestamp order, numeric values, and source-code coverage. It does not modify files in `data/raw/`.

Run it from the project root. The full source is about 1.1 GiB, so profiling takes a little time.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'etl' / 'profile_raw_data.py').exists():
    ROOT = ROOT.parent
subprocess.run([sys.executable, str(ROOT / 'etl' / 'profile_raw_data.py')], cwd=ROOT, check=True)
summary = json.loads((ROOT / 'reports' / 'raw_profile_summary.json').read_text(encoding='utf-8'))
print(f"Rows profiled: {summary['rows']:,}")
print(f"Files read: {summary['files']}")
print(f"Repeated incident IDs: {summary['logical_duplicate_incident_ids']:,}")

In [ ]:
missing = pd.read_csv(ROOT / 'reports' / 'missing_values.csv')
missing.loc[missing['blank_count'] > 0].sort_values('blank_count', ascending=False)

In [ ]:
numeric = pd.read_csv(ROOT / 'reports' / 'numeric_summary.csv')
timing_flags = pd.read_csv(ROOT / 'reports' / 'timing_by_validity_flag.csv')
display(numeric)
display(timing_flags)

In [ ]:
pd.DataFrame(summary['timestamp_order_checks']).T.assign(
    rows_with_any_pair_out_of_order=summary['rows_with_any_timestamp_order_issue']
)